# Full Sequence-to-Sequence Encoder-Decoder Transformer for Neural Machine Translation

## Team Members

- Poonam Biswal (2024ac05803)
- Vikas Mahadev Hiremath (2023ad05081)
- Praveen Kanwar (2024ac05746)
- S Amina (2024ac05758)

## Project

Neural Machine Translation using a full Encoder-Decoder Transformer implemented from scratch in PyTorch.

## 1. Introduction and Initialization
This notebook demonstrates the implementation of a complete Sequence-to-Sequence (Encoder-Decoder) Transformer from scratch using PyTorch. The goal is to build an English-to-German Neural Machine Translation (NMT) system.

First, we will import the required modules and initialize our environment.

In [1]:
import torch
import pandas as pd
import sys
import os

# Ensure src is in the python path
sys.path.append(os.path.abspath('..'))

from src.config import *
from src.data import load_and_preprocess_data, show_preprocessing_examples
from src.tokenizer_train import get_or_train_tokenizers
from src.masks import create_masks
from src.transformer import Seq2SeqTransformer
from src.utils import count_trainable_parameters

device = DEVICE
print(f"Using device: {device}")


Using device: mps


## 2. Dataset Description & Task 1: Dual-Language Preprocessing
We use the `bentrevett/multi30k` dataset from Hugging Face, which contains aligned English and German sentence pairs suitable for small-scale translation tasks.

Data preprocessing is crucial for NMT. We implement Unicode normalization, lowercasing, and specific punctuation handling for both English and German to ensure the model receives clean input.

In [2]:
print("Loading dataset...")
train_pairs, val_pairs, test_pairs = load_and_preprocess_data()
print(f"Loaded {len(train_pairs)} training, {len(val_pairs)} validation, and {len(test_pairs)} test pairs.")

print("\nShowing preprocessing examples...")
from datasets import load_dataset
raw_dataset = load_dataset(DATASET_NAME)
raw_train_pairs = [(ex[SRC_LANG], ex[TGT_LANG]) for ex in raw_dataset["train"]]
show_preprocessing_examples(raw_train_pairs, train_pairs, num_examples=3)

Loading dataset...
Loaded 29000 training, 1014 validation, and 1000 test pairs.

Showing preprocessing examples...
--- Preprocessing Examples ---
\nExample 1:
  Raw SRC:       Two young, White males are outside near many bushes.
  Processed SRC: two young , white males are outside near many bushes .
  Raw TGT:       Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.
  Processed TGT: zwei junge weiße männer sind im freien in der nähe vieler büsche .
\nExample 2:
  Raw SRC:       Several men in hard hats are operating a giant pulley system.
  Processed SRC: several men in hard hats are operating a giant pulley system .
  Raw TGT:       Mehrere Männer mit Schutzhelmen bedienen ein Antriebsradsystem.
  Processed TGT: mehrere männer mit schutzhelmen bedienen ein antriebsradsystem .
\nExample 3:
  Raw SRC:       A little girl climbing into a wooden playhouse.
  Processed SRC: a little girl climbing into a wooden playhouse .
  Raw TGT:       Ein kleines Mädchen klettert in ein 

## 3. Task 2: Shared vs Separate Tokenizer Training
We train separate BPE tokenizers for English and German. Although they share linguistic roots, they have significant differences in morphology, compounding behavior (especially German), and word formation. Separate vocabularies allow each tokenizer to optimally represent its language's unique structure.

In [3]:
print("Loading or training tokenizers...")
src_tokenizer, tgt_tokenizer = get_or_train_tokenizers(train_pairs)
print(f"Source vocabulary size: {src_tokenizer.get_vocab_size()}")
print(f"Target vocabulary size: {tgt_tokenizer.get_vocab_size()}")

Loading or training tokenizers...
Loading existing tokenizers...
Source vocabulary size: 8000
Target vocabulary size: 8000


## 4. Task 4: Positional and Padding Masking
Masking is essential:
- **Source Padding Mask**: Prevents the encoder and cross-attention from attending to `<pad>` tokens.
- **Target Padding Mask**: Similar to source padding mask but for the decoder.
- **Causal/Look-ahead Mask**: Prevents the decoder's self-attention from looking at future tokens during training, ensuring autoregressive generation.

In [4]:
print("Checking masks...")
# Create dummy data
dummy_src = torch.tensor([[1, 2, 3, PAD_IDX, PAD_IDX]]).to(device)
dummy_tgt = torch.tensor([[1, 4, 5, 6, PAD_IDX]]).to(device)

src_mask, tgt_mask = create_masks(dummy_src, dummy_tgt, device)
print(f"Source Mask Shape: {src_mask.shape}")
print(f"Target Mask Shape: {tgt_mask.shape}")
print(f"Target Mask (Causal + Padding):\n{tgt_mask[0, 0]}")

Checking masks...
Source Mask Shape: torch.Size([1, 1, 1, 5])
Target Mask Shape: torch.Size([1, 1, 5, 5])
Target Mask (Causal + Padding):
tensor([[ True, False, False, False, False],
        [ True,  True, False, False, False],
        [ True,  True,  True, False, False],
        [ True,  True,  True,  True, False],
        [ True,  True,  True,  True, False]], device='mps:0')


## 5. Task 3: Cross-Attention Mechanism & Architecture Initialization
The core of the Encoder-Decoder architecture is the Cross-Attention mechanism. In the decoder layers, the queries come from the previous decoder hidden state, while the keys and values come from the encoder's output. This allows the decoder to "attend" to relevant parts of the source sentence while generating the target sentence.

### Why Full Encoder-Decoder Architecture is Required for NMT
- **Encoder-only models (like BERT)** are designed for understanding context (e.g., classification) but lack an autoregressive generation mechanism.
- **Decoder-only models (like GPT)** are autoregressive but treat the source and target as a single sequence, which can be less effective at explicitly modeling the complex alignment between two distinct languages compared to a dedicated cross-attention mechanism.
- **Encoder-Decoder models** provide dedicated components: an encoder to build a rich representation of the source and a decoder that attends to this representation while generating the target sequence.

In [5]:
print("Initializing model...")
model = Seq2SeqTransformer(
    src_vocab_size=src_tokenizer.get_vocab_size(),
    tgt_vocab_size=tgt_tokenizer.get_vocab_size(),
    d_model=D_MODEL,
    n_heads=N_HEADS,
    d_ff=D_FF,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    dropout=DROPOUT,
    max_len=MAX_LEN
).to(device)

param_count = count_trainable_parameters(model)
print(f"Model initialized with {param_count:,} trainable parameters.")

Initializing model...
Model initialized with 10,105,664 trainable parameters.


## 6. Task 5 & 6: Training Loop, Evaluation, and Results
### Training Loop with Teacher Forcing
We use Teacher Forcing during training, where the decoder receives the true previous target token as input rather than its own prediction. We also employ the Noam learning rate scheduler, which increases the learning rate linearly for a warmup period and then decreases it proportionally to the inverse square root of the step number.

### Qualitative and Quantitative Translation Analysis
We evaluate the model using SacreBLEU, a standard metric for machine translation. We also perform qualitative analysis by comparing translations generated by Greedy Decoding and Beam Search.

The code block below will display the final results and sample translations if you have already run `train.py` and `evaluate.py`.

In [6]:
# Check if outputs exist
sample_csv_path = OUTPUT_DIR / "sample_translations.csv"
bleu_score_path = OUTPUT_DIR / "bleu_score.txt"

if not sample_csv_path.exists() or not bleu_score_path.exists():
    print("Output files are missing!")
    print("Please run `python train.py` and then `python evaluate.py` in your terminal first to generate the models and evaluations.")
else:
    with open(bleu_score_path, "r") as f:
        bleu_score = f.read().strip()
    print(f"Results found!\n{bleu_score}")
    
    print("\nSample Translations:")
    df = pd.read_csv(sample_csv_path)
    pd.set_option('display.max_colwidth', None)
    display(df.head())

Results found!
BLEU Score (Greedy Decoding): 22.66\n

Sample Translations:


,source_english,reference_german,greedy_translation,beam_translation,qualitative_comment
0,a man in an orange hat starring at something .,"ein mann mit einem orangefarbenen hut , der etwas anstarrt .","ein mann mit einem orangefarbenen hut , der etwas an einem blatt papier .","ein mann mit einem orangefarbenen hut , der etwas an einem blatt papier .",Partial semantic overlap with the reference; grammar may still be imperfect.
1,a boston terrier is running on lush green grass in front of a white fence .,ein boston terrier läuft über saftig-grünes gras vor einem weißen zaun .,ein weißer pudel läuft vor einem zaun auf grünem gras .,ein weiß gekleideter zaun läuft auf einer wiese vor einem zaun .,Partial semantic overlap with the reference; grammar may still be imperfect.
2,a girl in karate uniform breaking a stick with a front kick .,ein mädchen in einem karateanzug bricht ein brett mit einem tritt .,ein mädchen in einem trikot macht einen stock mit einem stock vor einem karate .,ein mädchen in arbeitskleidung macht einen stock mit einem stock .,Partial semantic overlap with the reference; grammar may still be imperfect.
3,"five people wearing winter jackets and helmets stand in the snow , with snowmobiles in the background .",fünf leute in winterjacken und mit helmen stehen im schnee mit schneemobilen im hintergrund .,"fünf personen mit helmen , winter helmen und schnee helmen stehen im hintergrund .","fünf personen mit helmen , winter helmen und schnee helmen stehen im hintergrund .",Partial semantic overlap with the reference; grammar may still be imperfect.
4,people are fixing the roof of a house .,leute reparieren das dach eines hauses .,die leute reparieren ein dach eines hauses .,leute reparieren das dach eines hauses .,Partial semantic overlap with the reference; grammar may still be imperfect.


## 7. Limitations & Future Work

### Limitations
- Small dataset limits generalization.
- Basic beam search implementation could be optimized.
- Fixed context length limits translation of very long sentences.

### Future Improvements
- Use larger datasets (e.g., WMT).
- Implement label smoothing.
- Add learning rate warmup and decay variations.

## 8. Conclusion
This project successfully demonstrates a full from-scratch implementation of the Transformer architecture for Neural Machine Translation.